In [13]:
import pandas as pd
import ast
import string
import numpy as np

In [3]:
remove_chars = string.punctuation + "؟\n"
translator = str.maketrans('', '', remove_chars)

val_df = pd.read_csv("../data/validation_translated.csv")
val_df.rename(columns={val_df.columns[0]: "real_dataset_index"}, inplace=True)

In [15]:
def n_probs(bigram: pd.DataFrame, sequence: list) -> float:
    if len(bigram) == 1:
        previous = ""
        unigram = True
    else:
        previous = "<START>"
        unigram = False
    
    probs = [] 
    
    sequence.append("<END>")
    for word in sequence:
        try:
            probs.append(bigram.loc[previous, word])
        except KeyError as _:
            probs.append(bigram.loc["<UNK>", "<UNK>"])
        if not unigram:
            previous = word
    
    return np.array(probs)

In [ ]:
context_bigram = pd.read_parquet("../data/bi_grams/context_bigram.parquet")

In [6]:
context_bigram

next_word,0,00,000,001,0011,00167,002,0023,0025,00253,...,참이슬,처음처럼,천국의,코레일,타블로,토석성,학생,한국과학기술연구원,한국철도공사,<UNK>
0,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,...,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021
00,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,...,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021
000,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,...,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021
001,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,...,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021
0011,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,...,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
토석성,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,...,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021
학생,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,...,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021
한국과학기술연구원,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,...,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021
한국철도공사,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,...,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021,0.000021


In [16]:
context_val = val_df["context_stripped"].apply(
    lambda x: [t.translate(translator).strip().lower() for t in ast.literal_eval(x) if t.translate(translator).strip()]
)

val_probs = []

for row in context_val:
    val_probs.append(n_probs(context_bigram, row))

In [17]:
print(val_probs[0])

[3.74433669e-05 4.24772747e-05 3.08493731e-02 4.92330017e-03
 3.25257496e-03 2.84829206e-03 7.68831010e-02 1.77238806e-03
 3.79570663e-04 5.58972654e-03 2.73638988e-05 4.24772747e-05
 3.08711839e-02 2.28026534e-04 7.85112568e-04 2.73638988e-05
 4.24808836e-05 6.35983973e-05 2.48094221e-02 2.83065513e-02
 2.00571118e-03 1.26137869e-04 1.90920662e-04 7.68831010e-02
 4.14593698e-05 4.24781768e-05 4.24412189e-05 2.25571835e-03
 1.03873360e-01 4.87147595e-04 6.36037908e-05 4.24205146e-05
 1.74561066e-02 1.14013267e-04 2.33575402e-04 1.03873360e-01
 4.14593698e-05 4.24781768e-05 2.97840655e-05 4.24547326e-05
 4.24088210e-05 3.85085000e-03 3.42048735e-04 4.22877683e-05
 4.24772747e-05 4.37163020e-04 1.39143762e-03 4.44322197e-04
 5.37746903e-03 2.73638988e-05 4.24808836e-05 1.86180341e-03
 2.19483039e-03 2.07296849e-05 4.24808836e-05 2.84829206e-03
 7.68831010e-02 8.26077944e-03 1.35292648e-02 2.41018928e-03
 3.08711839e-02 1.34742952e-04 1.27407470e-04 1.48229714e-04
 7.68831010e-02 4.871475